# After storm area impact mapping using Sentinel-2 data

Map the extent of storm affected areas using Sentinel-2 data.

Generate a pre event mosaic and use the first available post event image. 
Calculate the NDVI for both and generate a change image using an NDVI threshold and the formula (pre-image * 10) + post-image to generate the following classes for each image:

- 00 indicates no change
- 01 change
- 10 is not reasonable because it means that pre-event is true and post not true and can be disregarded
- 11 both images are true --> in both images NDVI is below the threshold and no change happened

In [1]:
import openeo
import rasterio
from openeo.processes import ProcessBuilder
from openeo.processes import quantiles
import leafmap
from shapely.geometry import shape
from folium.plugins import Draw
from IPython.display import JSON

In [3]:
# define properties for the outputs
from pathlib import Path

out_dir = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/wind/vaia")
#out_dir.mkdir()
event = "vaia_2018"

BANDS = ["B02", "B03", "B04", "B08", "SCL"]  

## 1) Define time frame and extent

In [4]:
# dates 
PRE_DATE  = ("2018-08-01", "2018-09-30")   
POST_DATE = ("2019-06-28", "2019-06-28")   

In [5]:
# open a map and zoom to the area of interest
m = leafmap.Map(center=(46.65, 11.4), zoom=8.5)
m

Map(center=[46.65, 11.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

In [6]:
feat = m.draw_features
geom_dict = feat[0]['geometry']
geom = shape(geom_dict)

minx, miny, maxx, maxy = geom.bounds

bbox = {
    "west": minx,
    "south": miny,
    "east": maxx,
    "north": maxy,
}

print(bbox)

{'west': 11.393509, 'south': 46.315636, 'east': 11.699753, 'north': 46.473335}


### 2) Authentificate and load the pre and post Sentinel-2 cube

In [7]:
connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()
connection.authenticate_oidc()

Authenticated using refresh token.
Authenticated using refresh token.


<Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>

In [8]:
# Load Sentinel-2 data
def load_s2(temporal_extent):
    cube = connection.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=bbox,
        temporal_extent=list(temporal_extent),
        bands=BANDS,
        max_cloud_cover=50,
    )
    return cube.resample_spatial(resolution=10, method="bilinear")

pre_cube=load_s2(PRE_DATE)
post_cube = load_s2(POST_DATE)

### 3) Create the mask from SCL

Create the mask using the following SCL values:

```
1  = SC_SATURATED_DEFECTIVE
3  = SC_CLOUD_SHADOW
7  = SC_CLOUD_LOW_PROBA / UNCLASSIFIED
8  = SC_CLOUD_MEDIUM_PROBA
9  = SC_CLOUD_HIGH_PROBA
10 = SC_THIN_CIRRUS
```

In [10]:
def mask_clouds(cube):
    scl = cube.band("SCL")
    return (
    (scl == 1) |
    (scl == 3) |
    (scl == 7) |
    (scl == 8) |
    (scl == 9) |
    (scl == 10)
    )

### 4) Generate the layer of valid observations

In [ ]:
post_cube_mask = mask_clouds(post_cube)
post_cube_masked = post_cube.mask(post_cube_mask)

In [ ]:
job = post_cube_masked.create_job()

In [ ]:
job.start_and_wait()

### 5) Generate monthly mosaic for pre event 

In [ ]:
file_post_cube = out_dir/ f"post_{event}.tif"

print(file_post_cube)

#post_cube_masked.download(file_post_cube, format='GTiff')
job.download_result(file_post_cube)

In [ ]:
pre_cube_mask = mask_clouds(pre_cube)

pre_cube_masked = pre_cube.mask(pre_cube_mask).reduce_dimension(
    dimension='t',
    reducer=lambda x: quantiles(data=x, probabilities=[0.25])
)

In [ ]:
job = pre_cube_masked.create_job()
job.start_and_wait()

In [ ]:
file_pre_cube = out_dir/ f"pre_mosaic_{event}.tif"

print(file_pre_cube)

#pre_cube_masked.download(file_pre_cube, format='GTiff')
job.download_result(file_pre_cube)

### 6) Calculate NDVI for pre and post event

In [ ]:
# calcualte NDVI 
def compute_ndvi(cube):
    nir = cube.band("B08")
    red = cube.band("B04")
    return (nir - red) / (nir + red)

post_ndvi = compute_ndvi(post_cube_masked)
pre_ndvi = compute_ndvi(pre_cube_masked)

In [ ]:
file_pre_ndvi = out_dir/ f"pre_mosaic_{event}_NDVI.tif"
file_post_ndvi = out_dir/ f"post_{event}_NDVI.tif"

pre_ndvi.download(file_pre_ndvi)
post_ndvi.download(file_post_ndvi)

### 7) Calculate Change between pre and post image

In [ ]:
# extract treshold + apply to each image and then classify image
thresh = 0.5

pre_ndvi_thresh  = (pre_ndvi  <= thresh).apply(lambda x: x * 1)  # cast bool → int
post_ndvi_thresh = (post_ndvi <= thresh).apply(lambda x: x * 1)   

change_ndvi = pre_ndvi_thresh * 10 + post_ndvi_thresh

# export
file_change_ndvi = out_dir/ f"change_{event}_NDVI.tif"

#change_ndvi.download(file_change_ndvi)

In [ ]:
job = change_ndvi.create_job()
job.start_and_wait()

In [ ]:
# download result

job.download_result(file_change_ndvi)